# 03 – Météo journalière

**Objectif :** récupérer, pour chaque commune et chaque jour de 2016 à 2025, 5 variables météo :

| Variable Open-Meteo | Sens |
|---|---|
| `temperature_2m_max` | température maximale (°C) |
| `relative_humidity_2m_min` | humidité relative minimale (%) |
| `wind_speed_10m_max` | vent maximal (km/h) |
| `precipitation_sum` | pluie (mm) |
| `et0_fao_evapotranspiration` | évapotranspiration (mm) : l'eau perdue par le sol et les plantes |

**Source :** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) (réanalyses ERA5 / ERA5-Land de Copernicus), gratuite et sans clé pour un usage non commercial.

## Étape 1 – Une session avec cache

`requests_cache` enregistre chaque réponse de l'API dans un fichier local (`data/raw/meteo_cache.sqlite`). Si le notebook plante en route, ou si on le relance, rien n'est retéléchargé. `expire_after=-1` : le cache n'expire jamais (la météo passée ne change plus).

In [1]:
import time
import pandas as pd
import requests_cache

session = requests_cache.CachedSession("../data/raw/meteo_cache", expire_after=-1)
communes = pd.read_csv("../data/processed/communes_13.csv", dtype={"code": str})

URL = "https://archive-api.open-meteo.com/v1/archive"
VARIABLES = "temperature_2m_max,relative_humidity_2m_min,wind_speed_10m_max,precipitation_sum,et0_fao_evapotranspiration"

## Étape 2 – Une fonction, testée sur un seul point

**Bonne pratique :** toujours tester sur **un seul** cas avant de lancer une boucle.

- `def` crée une **fonction** réutilisable : on écrit le code une fois, on l'appelle autant de fois qu'on veut ;
- `raise_for_status()` provoque une erreur claire si l'API répond mal (au lieu d'un échec silencieux) ;
- `timezone="Europe/Paris"` : les journées commencent à minuit heure française, comme les dates de la BDIFF.

In [2]:
def meteo_point(lat, lon):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "2016-01-01", "end_date": "2025-12-31",
        "daily": VARIABLES, "timezone": "Europe/Paris",
    }
    r = session.get(URL, params=params)
    r.raise_for_status()
    resultat = pd.DataFrame(r.json()["daily"])
    resultat.attrs["depuis_cache"] = r.from_cache   # True si la réponse vient du cache local
    return resultat

test = meteo_point(43.2803, 5.3806)   # Marseille
print(test.shape)   # attendu : (3653, 6)
test.head()

(3653, 6)


,time,temperature_2m_max,relative_humidity_2m_min,wind_speed_10m_max,precipitation_sum,et0_fao_evapotranspiration
0,2016-01-01,13.1,69,22.5,0.4,0.99
1,2016-01-02,13.4,76,25.3,6.8,0.95
2,2016-01-03,11.1,61,19.6,0.5,1.12
3,2016-01-04,12.8,69,15.3,0.7,0.88
4,2016-01-05,10.7,69,14.6,0.9,0.95


## Étape 3 – Le problème des limites de l'API (et sa solution)

La version gratuite d'Open-Meteo limite le nombre d'appels par minute, par heure et par jour, et une requête de **10 ans × 5 variables compte comme une centaine d'appels**. Une première boucle sur les 119 communes a été bloquée (erreur `429 Too Many Requests`) au bout de 6 communes, et n'aurait de toute façon pas tenu dans le quota quotidien.

**La solution : télécharger par « case » météo, pas par commune.** La météo ERA5-Land est calculée sur une grille d'environ **0,1° (≈ 10 km)**. Deux communes voisines dans la même case auraient la même météo : on arrondit les coordonnées au dixième de degré, on télécharge chaque case **une seule fois**, puis on redonne sa météo à chaque commune.

Le second `.round(1)` corrige les imprécisions des nombres à virgule (`43.300000000000004` → `43.3`).

In [3]:
communes["lat_case"] = ((communes["lat"] / 0.1).round() * 0.1).round(1)
communes["lon_case"] = ((communes["lon"] / 0.1).round() * 0.1).round(1)

cases = communes[["lat_case", "lon_case"]].drop_duplicates().reset_index(drop=True)
print(len(cases), "cases météo pour", len(communes), "communes")

52 cases météo pour 119 communes


**52 cases au lieu de 119 communes** : plus de deux fois moins d'appels, sans perte d'information.

## Étape 4 – Une boucle « patiente »

Si l'API refuse (429), on attend 60 secondes et on réessaie :
- `try` / `except` : *« essaie ceci ; si une erreur arrive, fais cela au lieu de planter »* ;
- `while True` répète jusqu'à ce que le `return` réussisse.

On fait aussi une pause de 2 secondes après chaque **vrai** appel à l'API (pas quand la réponse vient du cache) : `attrs` est un petit dictionnaire de « notes » qu'on peut attacher à un tableau pandas ; `meteo_point` y note si la réponse venait du cache.

Le premier téléchargement a pris plus d'une heure (pauses comprises). Grâce au cache, une ré-exécution prend quelques secondes.

In [4]:
def meteo_patiente(lat, lon):
    while True:
        try:
            return meteo_point(lat, lon)
        except Exception as e:
            print("   pause 60 s :", e)
            time.sleep(60)

resultats = []
for i, row in cases.iterrows():
    m = meteo_patiente(row["lat_case"], row["lon_case"])
    m["lat_case"] = row["lat_case"]
    m["lon_case"] = row["lon_case"]
    resultats.append(m)
    if not m.attrs["depuis_cache"]:   # vrai appel à l'API → petite pause de politesse
        time.sleep(2)

meteo_cases = pd.concat(resultats, ignore_index=True).rename(columns={"time": "jour"})
print(meteo_cases.shape)   # attendu : 52 × 3653 = (189956, 8)

(189956, 8)


`pd.concat` empile les 52 petits tableaux en un seul.

## Étape 5 – Vérifier, puis redonner la météo à chaque commune

In [5]:
meteo_cases.isna().sum()

jour                          0
temperature_2m_max            0
relative_humidity_2m_min      0
wind_speed_10m_max            0
precipitation_sum             0
et0_fao_evapotranspiration    0
lat_case                      0
lon_case                      0
dtype: int64

0 valeur manquante partout. On rattache maintenant chaque commune à la météo de sa case (jointure sur `lat_case` + `lon_case`) :

In [6]:
meteo = communes[["code", "lat_case", "lon_case"]].merge(meteo_cases, on=["lat_case", "lon_case"])
meteo = meteo.drop(columns=["lat_case", "lon_case"])
print(meteo.shape)   # attendu : (434707, 7), comme la table du notebook 02
meteo.to_csv("../data/processed/meteo_13.csv", index=False)

(434707, 7)


## Récap

- 5 variables météo journalières, 2016-2025, depuis Open-Meteo (ERA5).
- Limites de l'API → téléchargement par **case de 0,1°** (52 cases) avec **cache** et **boucle patiente**.
- **Limite à garder en tête :** des communes voisines partagent parfois exactement la même météo (résolution ≈ 10 km de la source).

**Prochaine étape (04) :** transformer cette météo brute en signaux de risque (feature engineering).